In [ ]:
#!/bin/bash
! curl -L -o heart-disease-risk-2026.zip\
  https://www.kaggle.com/api/v1/datasets/download/uditjain13/heart-disease-risk-2026

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  343k  100  343k    0     0   306k      0  0:00:01  0:00:01 --:--:--  306k


In [ ]:
!unzip heart-disease-risk-2026

Archive:  heart-disease-risk-2026.zip
  inflating: heart_disease_risk_2026.csv  


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/heart_disease_risk_2026.csv')

In [ ]:
df = df.drop('patient_id', axis=1)

In [ ]:
df.isnull().sum()

,0
age,0
sex,0
resting_bp_systolic,0
resting_bp_diastolic,0
cholesterol_total,0
hdl,0
ldl,0
triglycerides,0
fasting_blood_sugar,0
hba1c,0


In [ ]:
X = df.drop('has_heart_disease', axis=1)
y = df['has_heart_disease'].copy()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np


In [ ]:
cat_features = X_train.select_dtypes(exclude=[np.number]).columns
num_features = X_train.select_dtypes(include=[np.number]).columns



In [ ]:
num_pipelines = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipelines = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one-hot', OneHotEncoder(sparse_output=False, handle_unknown = 'ignore'))
])

transformer = ColumnTransformer([
    ('cat', cat_pipelines, cat_features),
    ('num', num_pipelines , num_features)
])

In [ ]:
X_train_transformer = transformer.fit_transform(X_train)
columns = transformer.get_feature_names_out()
X_train_transformed_df = pd.DataFrame(X_train_transformer, columns=columns)

In [ ]:
X_test_transformer = transformer.fit_transform(X_test)
X_test_transformed_df = pd.DataFrame(X_test_transformer, columns=columns)



In [ ]:
import torch

In [ ]:
X_train = torch.FloatTensor(X_train_transformer)
X_test = torch.FloatTensor(X_test_transformer)
y_train = torch.FloatTensor(np.array(y_train))
y_test = torch.FloatTensor(np.array(y_test))


In [ ]:
y_train = y_train.reshape(-1,1)
y_test = y_test.reshape(-1,1)


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size= 32, shuffle=True)


In [ ]:
import torch.nn as nn

In [ ]:
class MyCustomModel(nn.Module):
  def __init__(self, n_features, n_outputs=1):
    super().__init__()

    n_outputs = n_outputs if n_outputs > 2 else 1

    self.model = nn.Sequential(
        nn.Linear(n_features, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, n_outputs ),
        nn.ReLU()
    )
  def forward(self, X):
    return self.model(X)

In [ ]:
n_classes = len(np.unique(np.array(y_train)))

In [ ]:
n_features = X_train.shape[1]

In [ ]:
model = MyCustomModel(n_features=n_features, n_outputs=n_classes)

In [ ]:
model

MyCustomModel(
  (model): Sequential(
    (0): Linear(in_features=34, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): ReLU()
  )
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
def train(model, optimizer, criterion, train_loader, n_epochs, device):
  model.train()
  model.to(device)
  for epoch in  range(n_epochs):
    total_loss = 0.
    for X_batch, y_batch in train_loader:
      X_batch , y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = criterion(y_pred,y_batch)
      total_loss +=loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    mean_loss = total_loss / len(train_loader)
    print(f'Epoch: {epoch +1} / {n_epochs}, Loss:{mean_loss}')

In [ ]:
if torch.cuda.is_available:
  device = 'cuda'
elif torch.backends.mps.is_available:
  device = 'mps'
else:
  device = 'cpu'

In [ ]:
train(model, optimizer, criterion, train_loader, 20, device)

Epoch: 1 / 20, Loss:0.5868583556016286
Epoch: 2 / 20, Loss:0.5825987434387208
Epoch: 3 / 20, Loss:0.5805837033854591
Epoch: 4 / 20, Loss:0.5793083797560797
Epoch: 5 / 20, Loss:0.576862156788508
Epoch: 6 / 20, Loss:0.5742451747258505
Epoch: 7 / 20, Loss:0.5724095131291284
Epoch: 8 / 20, Loss:0.5707980802324083
Epoch: 9 / 20, Loss:0.5687070894241333
Epoch: 10 / 20, Loss:0.5662576009167565
Epoch: 11 / 20, Loss:0.5647913963264889
Epoch: 12 / 20, Loss:0.5630770991908179
Epoch: 13 / 20, Loss:0.5633012995455
Epoch: 14 / 20, Loss:0.559518494076199
Epoch: 15 / 20, Loss:0.5588438679112329
Epoch: 16 / 20, Loss:0.5576282999250624
Epoch: 17 / 20, Loss:0.5559289040830401
Epoch: 18 / 20, Loss:0.5551761962307824
Epoch: 19 / 20, Loss:0.5534935257169935
Epoch: 20 / 20, Loss:0.5506376704904768
